# Lightweight Context-Cluster — Full Ablation Study

**Chạy 4 ablation steps tuần tự trên Kaggle T4 GPU.**

| Step | Model | Flags | Mục tiêu |
|------|-------|-------|----------|
| 1 | ResNet-18 | — | Baseline mốc tham chiếu |
| 2 | CoC Baseline | PointShrink | Đo ảnh hưởng Point Shrink |
| 3 | CoC Baseline | PointShrink + Hamming | Thay Cosine bằng Hamming |
| 4 | HBCC Full | All flags | Lightweight CoC hoàn chỉnh |

**Ước tính thời gian trên T4:** ~1–1.5h/step × 4 = **4–6 giờ tổng**

---
### Cách setup (chọn 1 trong 2):
- **Cách A — GitHub** *(khuyên dùng)*: Push code lên GitHub, dùng cell 2A
- **Cách B — Kaggle Dataset**: Zip folder → Upload Kaggle Dataset, dùng cell 2B

## Bước 1 — Kiểm tra GPU & Cài thư viện

In [ ]:
import subprocess, os, sys, json, time
import torch

os.system('nvidia-smi --query-gpu=name,memory.total --format=csv,noheader')
os.system('pip install einops pyyaml -q')

print(f'PyTorch  : {torch.__version__}')
print(f'CUDA     : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU      : {torch.cuda.get_device_name(0)}')
    print(f'VRAM     : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

## Bước 2A — Setup Code từ GitHub

> Đổi `GITHUB_REPO` thành URL repo của bạn rồi chạy cell này.  
> Bỏ qua nếu dùng Cách B.

In [ ]:
GITHUB_REPO = 'https://github.com/YOUR-USERNAME/Lightweight_Context-Cluster.git'
CODE_DIR    = '/kaggle/working/code'

if not os.path.exists(CODE_DIR):
    os.system(f'git clone {GITHUB_REPO} {CODE_DIR}')
else:
    os.system(f'cd {CODE_DIR} && git pull')

os.chdir(CODE_DIR)
sys.path.insert(0, CODE_DIR)
print('Working dir:', os.getcwd())
print('Files:', sorted(os.listdir('.')))

## Bước 2B — Setup Code từ Kaggle Dataset

Cách tạo Kaggle Dataset:
1. Trong terminal local: `cd ~/Documents/Research && zip -r hbcc-code.zip Lightweight_Context-Cluster/src Lightweight_Context-Cluster/configs -x '*.pyc'`
2. Vào [kaggle.com/datasets/new](https://www.kaggle.com/datasets/new) → Upload `hbcc-code.zip`
3. Trong notebook: **+ Add data** → tìm dataset → Add
4. Chạy cell bên dưới (đổi slug nếu cần)

In [ ]:
import zipfile, glob

DATASET_SLUG = 'hbcc-code'  # đổi thành slug dataset của bạn
DATASET_PATH = f'/kaggle/input/{DATASET_SLUG}/'
CODE_DIR     = '/kaggle/working/code'

os.makedirs(CODE_DIR, exist_ok=True)
zip_files = glob.glob(f'{DATASET_PATH}*.zip')

if zip_files:
    print(f'Extracting {zip_files[0]}...')
    with zipfile.ZipFile(zip_files[0]) as zf:
        zf.extractall(CODE_DIR)
    # Tìm root của project sau khi giải nén
    subdirs = [d for d in os.listdir(CODE_DIR) if os.path.isdir(os.path.join(CODE_DIR, d))]
    if subdirs:
        CODE_DIR = os.path.join(CODE_DIR, subdirs[0])
else:
    CODE_DIR = DATASET_PATH

os.chdir(CODE_DIR)
sys.path.insert(0, CODE_DIR)
print('Working dir:', os.getcwd())
print('Files:', sorted(os.listdir('.')))

## Bước 3 — Tạo thư mục & Tải CIFAR-10

In [ ]:
import torchvision

for d in ['/kaggle/working/experiments', '/kaggle/working/results', '/kaggle/working/data']:
    os.makedirs(d, exist_ok=True)

# Tải CIFAR-10 (~170MB). Cần bật Internet trong Settings của notebook.
print('Downloading CIFAR-10 (if not cached)...')
ds = torchvision.datasets.CIFAR10(root='/kaggle/working/data', train=True, download=True)
print(f'CIFAR-10 OK: {len(ds):,} training samples')

# Kiểm tra configs
print('\nKaggle configs:')
for step in ['step1_baseline', 'step2_shrink', 'step3_hbcc', 'step4_full']:
    p = f'configs/kaggle/{step}.yaml'
    print(f'  {"OK" if os.path.exists(p) else "MISSING"}: {p}')

## Bước 4 — Helper: run_training + show_result

In [ ]:
def run_training(config_name: str, label: str) -> bool:
    """Chạy 1 ablation step, in log real-time."""
    config_path = f'configs/kaggle/{config_name}.yaml'
    print(f'\n{"="*60}\n  {label}\n  Start: {time.strftime("%H:%M:%S")}\n{"="*60}\n')
    t0 = time.time()
    proc = subprocess.Popen(
        [sys.executable, 'src/train.py', '--config', config_path],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
    )
    for line in proc.stdout:
        print(line, end='', flush=True)
    proc.wait()
    elapsed = (time.time() - t0) / 60
    ok = proc.returncode == 0
    print(f'\n[{label}] {"DONE" if ok else "FAILED"} — {elapsed:.1f} min')
    return ok

def show_result(path: str):
    """In kết quả từ file JSON."""
    try:
        r = json.load(open(path))
        for k, v in [
            ('Model',      r.get('model')),
            ('Val Acc@1',  f"{r.get('best_val_acc1', 0):.2f}%"),
            ('Params',     f"{r.get('total_params', 0):,}"),
            ('FLOPs',      f"{r.get('flops', 0):,}"),
            ('Throughput', f"{r.get('throughput', 0):.1f} img/s"),
            ('Latency',    f"{r.get('latency_ms', 0):.2f} ms"),
        ]:
            print(f'  {k:<12}: {v}')
    except FileNotFoundError:
        print(f'  Chưa có: {path}')

print('Helpers ready.')

## (Tuỳ chọn) Quick Test — 5 epochs

> **Chạy cell này TRƯỚC** để kiểm tra pipeline (~3 phút). Nếu pass thì mới chạy full.

In [ ]:
import yaml

quick_cfg = {
    'model': 'hbcc', 'dataset': 'cifar10',
    'data_dir': '/kaggle/working/data', 'num_classes': 10, 'img_size': 32,
    'batch_size': 256, 'num_workers': 2, 'pin_memory': True,
    'use_random_crop': True, 'use_random_flip': True, 'use_mixup': False,
    'optimizer': 'adamw', 'lr': 0.001, 'weight_decay': 0.05,
    'scheduler': 'cosine', 'warmup_epochs': 1, 'min_lr': 1e-6, 'epochs': 5,
    'device': 'cuda', 'amp': True,
    'use_point_shrink': True, 'use_hamming': True,
    'use_linear_bottleneck': True, 'use_channel_shuffle': True,
    'coc_heads': 4, 'coc_head_dim': 16, 'coc_proposal_w': 2, 'coc_proposal_h': 2,
    'coc_fold_w': 1, 'coc_fold_h': 1,
    'log_dir': '/kaggle/working/experiments', 'save_dir': '/kaggle/working/experiments',
    'log_interval': 10, 'seed': 42,
    'result_file': '/kaggle/working/results/quicktest.json',
}
os.makedirs('configs/kaggle', exist_ok=True)
with open('configs/kaggle/quicktest.yaml', 'w') as f:
    yaml.dump(quick_cfg, f)

run_training('quicktest', 'QUICK TEST — 5 epochs HBCC Full')

## Step 1 — ResNet-18 Baseline

In [ ]:
ok1 = run_training('step1_baseline', 'STEP 1 — ResNet-18 Baseline')

In [ ]:
show_result('/kaggle/working/results/step1_baseline.json')

## Step 2 — CoC Baseline + Point Shrink

In [ ]:
ok2 = run_training('step2_shrink', 'STEP 2 — CoC + Point Shrink')

In [ ]:
show_result('/kaggle/working/results/step2_coc_shrink.json')

## Step 3 — CoC + Point Shrink + Hamming

In [ ]:
ok3 = run_training('step3_hbcc', 'STEP 3 — CoC + Hamming Distance')

In [ ]:
show_result('/kaggle/working/results/step3_coc_hamming.json')

## Step 4 — HBCC Full (tất cả kỹ thuật)

In [ ]:
ok4 = run_training('step4_full', 'STEP 4 — HBCC Full')

In [ ]:
show_result('/kaggle/working/results/step4_hbcc_full.json')

## Tổng kết — So sánh tất cả steps

In [ ]:
steps = [
    ('Step 1 — ResNet-18',   '/kaggle/working/results/step1_baseline.json'),
    ('Step 2 — CoC+Shrink',  '/kaggle/working/results/step2_coc_shrink.json'),
    ('Step 3 — CoC+Hamming', '/kaggle/working/results/step3_coc_hamming.json'),
    ('Step 4 — HBCC Full',   '/kaggle/working/results/step4_hbcc_full.json'),
]

print(f'{"Step":<24} {"Acc@1":>8} {"Params":>12} {"FLOPs":>12} {"img/s":>8}')
print('-' * 68)
for label, path in steps:
    try:
        r = json.load(open(path))
        print(f'{label:<24} {r.get("best_val_acc1",0):>7.2f}% '
              f'{r.get("total_params",0):>12,} '
              f'{r.get("flops",0):>12,} '
              f'{r.get("throughput",0):>8.0f}')
    except FileNotFoundError:
        print(f'{label:<24}  [chưa có kết quả]')
print('-' * 68)
print('Checkpoints:', os.listdir('/kaggle/working/experiments/')[:5])